# Train on Kaggle

1. **Settings:** set the accelerator to GPU T4 ×2 (or P100) and turn the internet on.
2. **Add data → the team's VOC dataset.** It must contain a folder with `VOC2007/` and `VOC2012/` inside.
3. Set `RUN` and `CONFIG` below, then **Save Version → Save & Run All**, so it keeps running with the browser closed.
4. Outputs land in `/kaggle/working/runs/<RUN>/` (`best.pt`, `last.pt`, `history.csv`, `loss_curve.png`).

**Resuming after the 12 h limit:** add the previous version's output as an input dataset, set `RESUME_FROM` to its `runs/<RUN>` folder, and run again.

In [ ]:
RUN = "A_centernet"                          # A_centernet | B_noaug | C_grid
CONFIG = "configs/centernet_voc5.yaml"       # centernet_voc5.yaml | centernet_voc5_noaug.yaml | grid_voc5.yaml
EXTRA = ""                                   # e.g. "train.epochs=60" (B and C) or "input_size=512"
RESUME_FROM = ""                             # e.g. "/kaggle/input/<previous-output>/runs/A_centernet"
BRANCH = "main"

In [ ]:
import os
REPO_DIR = "/kaggle/working/csce4603-object-detection"
if not os.path.exists(REPO_DIR):
    !git clone -q -b $BRANCH https://github.com/Abdallah-Afifi/csce4603-object-detection.git $REPO_DIR
else:
    !git -C $REPO_DIR pull -q
%cd $REPO_DIR
!git log -1 --oneline
!pip install -q pyyaml pytest
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Find the VOC root: a folder that contains both VOC2007/ and VOC2012/ (searched a few levels deep only)
def find_voc_root(base="/kaggle/input", max_depth=5):
    for dirpath, dirnames, _ in os.walk(base):
        if dirpath[len(base):].count(os.sep) >= max_depth:
            dirnames[:] = []
            continue
        if {"VOC2007", "VOC2012"} <= set(dirnames):
            return dirpath
    return None

VOC_ROOT = find_voc_root()
print("VOC root:", VOC_ROOT)
assert VOC_ROOT, "Add the VOC dataset (with VOC2007/ and VOC2012/ in one folder) to this notebook"

In [ ]:
# Fast checks: unit tests (CPU) and a model-only overfit check on 8 real images (~1 min on GPU)
!python -m pytest -q -x
!python scripts/overfit_check.py --config $CONFIG --set data.voc_root=$VOC_ROOT --steps 300

In [ ]:
OUT = f"/kaggle/working/runs/{RUN}"
if RESUME_FROM and not os.path.exists(f"{OUT}/last.pt"):
    os.makedirs("/kaggle/working/runs", exist_ok=True)
    !cp -r $RESUME_FROM $OUT
    print("resuming from", RESUME_FROM)
!python scripts/train.py --config $CONFIG --out $OUT --set data.voc_root=$VOC_ROOT $EXTRA

In [ ]:
from IPython.display import Image, display
display(Image(f"{OUT}/loss_curve.png"))
import pandas as pd
pd.read_csv(f"{OUT}/history.csv").tail(10)